# Counterfactual Dynamical Analysis for AKOrN Models

This notebook performs dynamical analysis on selected cases from the parameter sweep using the new `AKOrNDynamicalAnalyzer` class. We analyze energy dynamics, temporal evolution, and convergence properties across different gamma and T values.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os
import json
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Add project root to path (since we're in notebooks/)
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Setup imports
from source.models.classification.my_knet import MyAKOrN
from source.models.classification.analysis_utils import AKOrNDynamicalAnalyzer
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Project root: {project_root}")

In [ ]:
# Select interesting cases from the sweep analysis
# Based on diversity in gamma and T values (corrected from actual sweep results)
selected_cases = [
    {"dir": "sweep_20250708_581384.opbs_0", "gamma": 0.01, "T": 3, "name": "Very low gamma, very low T"},
    {"dir": "sweep_20250708_581385.opbs_1", "gamma": 0.1, "T": 3, "name": "Low gamma, very low T"},
    {"dir": "sweep_20250708_581389.opbs_5", "gamma": 1.0, "T": 7, "name": "High gamma, low T"},
    {"dir": "sweep_20250708_581394.opbs_10", "gamma": 0.1, "T": 31, "name": "Low gamma, high T"},
    {"dir": "sweep_20250708_581398.opbs_14", "gamma": 1.0, "T": 63, "name": "High gamma, very high T"}
]

print(f"Selected {len(selected_cases)} cases for dynamical analysis:")
for case in selected_cases:
    print(f"  {case['name']}: gamma={case['gamma']}, T={case['T']}")

In [ ]:
def load_model_from_sweep(sweep_dir):
    """Load a trained model from sweep results."""
    results_dir = project_root / "results"
    
    # Load config
    config_path = results_dir / sweep_dir / "parameters.json"
    if not config_path.exists():
        print(f"Config not found for {sweep_dir}")
        return None
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    # Check if model exists
    model_path = results_dir / sweep_dir / "my_akorn_cifar10_final.pth"
    if not model_path.exists():
        print(f"Model not found for {sweep_dir}")
        return None
    
    try:
        # Create model
        model = MyAKOrN(
            n=config['n'],
            ch=config['ch'], 
            out_classes=config['num_classes'],
            L=config['L'],
            T=config['T'],
            J=config['J'],
            J_bias=config['J_bias'],
            ksizes=config['ksizes'],
            ro_ksize=config['ro_ksize'],
            ro_N=config['ro_N'],
            norm=config['norm'],
            c_norm=config['c_norm'],
            gamma=config['gamma'],
            use_omega=config['use_omega'],
            init_omg=config['init_omg'],
            global_omg=config['global_omg'],
            learn_omg=config['learn_omg'],
            ensemble=config['ensemble']
        ).to(device)
        
        # Load weights
        checkpoint = torch.load(model_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        
        print(f"Successfully loaded {sweep_dir}")
        return model, config
        
    except Exception as e:
        print(f"Error loading {sweep_dir}: {e}")
        return None

In [ ]:
# Load selected models
loaded_models = {}

for case in selected_cases:
    result = load_model_from_sweep(case["dir"])
    if result is not None:
        model, config = result
        loaded_models[case["name"]] = {
            "model": model,
            "config": config,
            "gamma": case["gamma"],
            "T": case["T"],
            "sweep_dir": case["dir"]
        }

print(f"\nSuccessfully loaded {len(loaded_models)} models for dynamical analysis")

In [ ]:
def create_simple_test_loader(batch_size=32, data_dir='./data'):
    """Create a simple test data loader for CIFAR10."""
    transform_test = transforms.Compose([
        transforms.ToTensor(),
    ])

    test_dataset = CIFAR10(
        root=data_dir, 
        train=False, 
        download=True, 
        transform=transform_test
    )

    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=2,
        pin_memory=True
    )
    
    return test_loader

# Load test data for analysis
test_loader = create_simple_test_loader(batch_size=32, data_dir=str(project_root / 'data'))

# Get a sample input for analysis
test_batch = next(iter(test_loader))
test_input, test_labels = test_batch
test_input = test_input.to(device)

# Use first sample for detailed analysis
sample_input = test_input[0:1]  # Single sample
print(f"Sample input shape: {sample_input.shape}")
print(f"Sample label: {test_labels[0].item()}")

## Single Layer Dynamical Analysis

First, let's analyze the energy dynamics for layer 0 across different models.

In [ ]:
# Analyze energy dynamics for layer 0 across all models
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (name, model_data) in enumerate(loaded_models.items()):
    if idx >= len(axes):
        break
        
    model = model_data["model"]
    gamma = model_data["gamma"]
    T = model_data["T"]
    
    # Create dynamical analyzer for layer 0
    analyzer = AKOrNDynamicalAnalyzer(model, layer_idx=0, device=device)
    
    # Extract energy dynamics
    energy_data = analyzer.extract_energy_dynamics(sample_input)
    trajectory = energy_data[0]['trajectory']
    
    # Plot
    ax = axes[idx]
    time_steps = range(len(trajectory))
    ax.plot(time_steps, trajectory, marker='o', linewidth=2, color='blue')
    ax.set_xlabel('Time Step', fontsize=12)
    ax.set_ylabel('Energy', fontsize=12)
    ax.set_title(f'{name}\n(γ={gamma}, T={T})', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    # Add final energy value
    final_energy = trajectory[-1]
    ax.text(len(trajectory)-1, final_energy, f'{final_energy:.3f}', 
            fontsize=9, ha='left', va='bottom')

# Hide unused subplots
for idx in range(len(loaded_models), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Energy Dynamics Comparison - Layer 0', fontsize=16)
plt.tight_layout()
plt.show()

## Multi-Layer Comparative Analysis

Now let's analyze multiple layers simultaneously for one interesting case.

In [ ]:
# Select the "Mid gamma, high T" case for multi-layer analysis
selected_case_name = "Mid gamma, high T"
if selected_case_name in loaded_models:
    model_data = loaded_models[selected_case_name]
    model = model_data["model"]
    
    # Create multi-layer analyzer
    multi_analyzer = AKOrNDynamicalAnalyzer(model, layer_idx=[0, 1, 2], device=device)
    
    # Plot energy dynamics for all layers
    multi_analyzer.plot_energy_dynamics(sample_input, show=True, 
                                      title=f"Multi-Layer Energy Dynamics: {selected_case_name}")
else:
    print(f"Case '{selected_case_name}' not available")

## T-Value Comparison Analysis

Let's see how different T values affect the energy dynamics for one of our models.

In [ ]:
# T-value comparison for the "Low gamma, moderate T" case
selected_case_name = "Low gamma, moderate T"
if selected_case_name in loaded_models:
    model_data = loaded_models[selected_case_name]
    model = model_data["model"]
    
    # Create analyzer for layer 0
    analyzer = AKOrNDynamicalAnalyzer(model, layer_idx=0, device=device)
    
    # Compare different T values
    T_values = [4, 8, 12, 16, 20]
    print(f"Comparing T values {T_values} for {selected_case_name}...")
    
    analyzer.plot_T_comparison(sample_input, T_values, show=True)
else:
    print(f"Case '{selected_case_name}' not available")

## Multi-Layer T-Value Comparison

Let's do a more comprehensive analysis with multiple layers and T values.

In [ ]:
# Multi-layer T-value comparison for "High gamma, low T" case
selected_case_name = "High gamma, low T"
if selected_case_name in loaded_models:
    model_data = loaded_models[selected_case_name]
    model = model_data["model"]
    
    # Create multi-layer analyzer
    multi_analyzer = AKOrNDynamicalAnalyzer(model, layer_idx=[0, 1, 2], device=device)
    
    # Compare T values across layers
    T_values = [4, 8, 12]
    print(f"Multi-layer T-value comparison for {selected_case_name}...")
    
    multi_analyzer.plot_T_comparison(sample_input, T_values, show=True)
else:
    print(f"Case '{selected_case_name}' not available")

## Convergence Analysis

Let's analyze convergence properties across different T values.

In [ ]:
# Convergence analysis for "Very high gamma, moderate-high T" case
selected_case_name = "Very high gamma, moderate-high T"
if selected_case_name in loaded_models:
    model_data = loaded_models[selected_case_name]
    model = model_data["model"]
    
    # Create analyzer for all layers
    analyzer = AKOrNDynamicalAnalyzer(model, layer_idx=[0, 1, 2], device=device)
    
    # Run comparison first to get convergence data
    T_values = [6, 10, 14, 18]
    comparison_results = analyzer.compare_T_values(sample_input, T_values)
    
    # Plot convergence analysis
    analyzer.plot_convergence_analysis(T_values, show=True)
else:
    print(f"Case '{selected_case_name}' not available")

## Comprehensive Dynamical Reports

Let's generate comprehensive reports for each model.

In [ ]:
# Generate comprehensive reports for all models
reports = {}
T_values_for_comparison = [4, 8, 12, 16]

for name, model_data in loaded_models.items():
    print(f"\nGenerating report for {name}...")
    
    model = model_data["model"]
    gamma = model_data["gamma"]
    T = model_data["T"]
    
    # Create analyzer for layer 0
    analyzer = AKOrNDynamicalAnalyzer(model, layer_idx=0, device=device)
    
    # Generate report
    report = analyzer.generate_dynamical_report(
        sample_input, 
        T_values=T_values_for_comparison,
        save_path=f"results/dynamical_reports/{name.replace(' ', '_')}_report.json"
    )
    
    reports[name] = report
    
    # Print key statistics
    print(f"  Original T: {report['model_info']['original_T']}")
    print(f"  Analyzed layers: {report['model_info']['analyzed_layers']}")
    
    # Summary statistics
    for T_key, stats in report['summary_statistics'].items():
        T_val = T_key.replace('T_', '')
        success_rate = stats['convergence_success_rate']
        avg_energy = stats['average_final_energy']
        print(f"  T={T_val}: Success rate={success_rate:.2f}, Avg final energy={avg_energy:.4f}")

print(f"\nGenerated reports for {len(reports)} models.")

## Cross-Model Comparison

Let's compare the dynamical properties across different gamma values.

In [ ]:
# Cross-model comparison: Final energies vs gamma values
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Collect data for comparison
gamma_values = []
final_energies_T8 = []
final_energies_T12 = []
final_energies_T16 = []
convergence_rates_T8 = []
model_names = []

for name, report in reports.items():
    model_data = loaded_models[name]
    gamma = model_data["gamma"]
    
    gamma_values.append(gamma)
    model_names.append(name)
    
    # Extract final energies for different T values
    if 'T_8' in report['summary_statistics']:
        final_energies_T8.append(report['summary_statistics']['T_8']['average_final_energy'])
        convergence_rates_T8.append(report['summary_statistics']['T_8']['convergence_success_rate'])
    else:
        final_energies_T8.append(np.nan)
        convergence_rates_T8.append(np.nan)
    
    if 'T_12' in report['summary_statistics']:
        final_energies_T12.append(report['summary_statistics']['T_12']['average_final_energy'])
    else:
        final_energies_T12.append(np.nan)
    
    if 'T_16' in report['summary_statistics']:
        final_energies_T16.append(report['summary_statistics']['T_16']['average_final_energy'])
    else:
        final_energies_T16.append(np.nan)

# Plot 1: Final energies vs gamma for different T values
axes[0,0].scatter(gamma_values, final_energies_T8, label='T=8', alpha=0.7, s=60)
axes[0,0].scatter(gamma_values, final_energies_T12, label='T=12', alpha=0.7, s=60)
axes[0,0].scatter(gamma_values, final_energies_T16, label='T=16', alpha=0.7, s=60)
axes[0,0].set_xlabel('Gamma')
axes[0,0].set_ylabel('Average Final Energy')
axes[0,0].set_title('Final Energy vs Gamma')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)
axes[0,0].set_xscale('log')

# Plot 2: Convergence rates vs gamma
axes[0,1].scatter(gamma_values, convergence_rates_T8, color='blue', alpha=0.7, s=60)
axes[0,1].set_xlabel('Gamma')
axes[0,1].set_ylabel('Convergence Success Rate (T=8)')
axes[0,1].set_title('Convergence Rate vs Gamma')
axes[0,1].grid(True, alpha=0.3)
axes[0,1].set_xscale('log')
axes[0,1].set_ylim(0, 1.1)

# Plot 3: Energy dynamics comparison for T=8
for name, model_data in loaded_models.items():
    model = model_data["model"]
    gamma = model_data["gamma"]
    
    analyzer = AKOrNDynamicalAnalyzer(model, layer_idx=0, device=device)
    energy_data = analyzer.extract_energy_dynamics(sample_input)
    trajectory = energy_data[0]['trajectory']
    
    axes[1,0].plot(trajectory, label=f'γ={gamma}', alpha=0.7, linewidth=2)

axes[1,0].set_xlabel('Time Step')
axes[1,0].set_ylabel('Energy')
axes[1,0].set_title('Energy Trajectories Comparison (Original T)')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Plot 4: Model summary table (text)
axes[1,1].axis('off')
table_data = []
for name, model_data in loaded_models.items():
    gamma = model_data["gamma"]
    T = model_data["T"]
    final_energy = final_energies_T8[model_names.index(name)]
    conv_rate = convergence_rates_T8[model_names.index(name)]
    
    table_data.append([f'γ={gamma}', f'T={T}', f'{final_energy:.4f}', f'{conv_rate:.2f}'])

axes[1,1].table(cellText=table_data,
               colLabels=['Gamma', 'Original T', 'Final Energy (T=8)', 'Conv. Rate'],
               cellLoc='center',
               loc='center')
axes[1,1].set_title('Model Summary')

plt.tight_layout()
plt.show()

## Model Performance vs Dynamical Properties

Let's see if there's a relationship between dynamical properties and model performance.

In [ ]:
# Evaluate model accuracy for a subset of test data
accuracy_results = {}

print("Evaluating model accuracies...")
for name, model_data in loaded_models.items():
    model = model_data["model"]
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(test_loader):
            if batch_idx >= 10:  # Limit to first 10 batches for speed
                break
                
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            total += target.size(0)
    
    accuracy = correct / total
    accuracy_results[name] = accuracy
    print(f"  {name}: {accuracy*100:.2f}%")

# Plot accuracy vs dynamical properties
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

accuracies = [accuracy_results[name] for name in model_names]

# Accuracy vs final energy
axes[0].scatter(final_energies_T8, accuracies, alpha=0.7, s=100)
for i, name in enumerate(model_names):
    axes[0].annotate(f'γ={gamma_values[i]}', 
                    (final_energies_T8[i], accuracies[i]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
axes[0].set_xlabel('Final Energy (T=8)')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Accuracy vs Final Energy')
axes[0].grid(True, alpha=0.3)

# Accuracy vs convergence rate
axes[1].scatter(convergence_rates_T8, accuracies, alpha=0.7, s=100)
for i, name in enumerate(model_names):
    axes[1].annotate(f'γ={gamma_values[i]}', 
                    (convergence_rates_T8[i], accuracies[i]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
axes[1].set_xlabel('Convergence Success Rate (T=8)')
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Accuracy vs Convergence Rate')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nAnalysis complete!")

## Summary

This notebook demonstrated comprehensive dynamical analysis of AKOrN models using the `AKOrNDynamicalAnalyzer` class:

1. **Energy Dynamics**: Analyzed energy trajectories across different models and layers
2. **Multi-Layer Analysis**: Compared dynamics across multiple layers simultaneously
3. **T-Value Effects**: Investigated how different T values affect convergence and dynamics
4. **Convergence Properties**: Analyzed convergence rates and timing across different configurations
5. **Cross-Model Comparison**: Compared dynamical properties across different gamma values
6. **Performance Correlation**: Explored relationships between dynamical properties and model performance

The analysis reveals how the gamma parameter affects both the dynamical behavior and convergence properties of AKOrN models, providing insights into the relationship between oscillator coupling strength and model dynamics.

## Summary: Network Analysis of Connectivity Decomposition

This analysis applies network science methods to the connectivity decomposition variables (frob_norms, c_R, c_S) from the AKOrN models:

### **Key Findings:**

1. **Decomposition Variables as Network Nodes**: We treated each connectivity block's decomposition metrics as nodes in a network, creating coupling matrices based on similarity in frob_norms, c_R, and c_S values.

2. **Multiple Coupling Methods**: Compared correlation-based, distance-based, product-based, and adjacency-based methods for creating networks from decomposition variables.

3. **Network Metrics Reveal Structure**: The resulting networks show different spectral properties (λ₂, eigenratio) and community structure (modularity) depending on the γ and T parameters.

4. **Parameter Dependencies**: Network metrics derived from decomposition variables show correlations with the underlying AKOrN parameters, providing another lens for understanding model organization.

### **Insights for Kuramoto Dynamics:**

- **Frobenius norms** reflect overall coupling strength and correlate with network connectivity
- **c_R and c_S components** capture rotational vs symmetric coupling patterns 
- **Network metrics** provide global measures of synchronization potential based on local decomposition

This network perspective on connectivity decomposition variables offers a bridge between local coupling analysis and global network dynamics in AKOrN models.

In [ ]:
def analyze_decomposition_variable_relationships(connectivity_decomposition_data):
    """Analyze relationships between decomposition variables and network metrics."""
    
    # Collect all data across models and layers
    all_data = []
    
    for name, data in connectivity_decomposition_data.items():
        gamma = data['gamma']
        T = data['T']
        
        for layer_idx, decomp in data['decomposition'].items():
            # Create coupling matrix and compute metrics
            coupling_matrix = create_coupling_matrix_from_decomposition(
                decomp['frob_norms'], decomp['c_R'], decomp['c_S'], method='correlation'
            )
            
            try:
                metrics = compute_all_metrics(coupling_matrix, directed=False, threshold=1e-6)
                
                all_data.append({
                    'model': name,
                    'layer': layer_idx,
                    'gamma': gamma,
                    'T': T,
                    'frob_mean': decomp['frob_norms'].mean(),
                    'frob_std': decomp['frob_norms'].std(),
                    'cR_mean': decomp['c_R'].mean(),
                    'cR_std': decomp['c_R'].std(),
                    'cS_mean': decomp['c_S'].mean(),
                    'cS_std': decomp['c_S'].std(),
                    'lambda_2': metrics['spectral']['lambda_2'],
                    'eigenratio': metrics['spectral']['eigenratio'],
                    'modularity': metrics['community']['modularity'],
                    'avg_path': metrics['path']['avg_shortest_path']
                })
            except:
                continue
    
    # Convert to DataFrame for analysis
    df = pd.DataFrame(all_data)
    print(f"Collected data for {len(df)} layer-model combinations")
    
    # Create correlation matrix
    numerical_cols = ['gamma', 'T', 'frob_mean', 'frob_std', 'cR_mean', 'cR_std', 
                     'cS_mean', 'cS_std', 'lambda_2', 'eigenratio', 'modularity', 'avg_path']
    
    correlation_matrix = df[numerical_cols].corr()
    
    # Plot correlation heatmap
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    # Correlation heatmap
    im = axes[0].imshow(correlation_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    axes[0].set_xticks(range(len(numerical_cols)))
    axes[0].set_yticks(range(len(numerical_cols)))
    axes[0].set_xticklabels(numerical_cols, rotation=45, ha='right')
    axes[0].set_yticklabels(numerical_cols)
    axes[0].set_title('Correlation Matrix: Decomposition Variables vs Network Metrics')
    
    # Add correlation values to heatmap
    for i in range(len(numerical_cols)):
        for j in range(len(numerical_cols)):
            text = axes[0].text(j, i, f'{correlation_matrix.iloc[i, j]:.2f}',
                               ha="center", va="center", color="black", fontsize=8)
    
    plt.colorbar(im, ax=axes[0])
    
    # Scatter plot: frob_norms vs network connectivity
    colors = plt.cm.viridis(df['gamma'] / df['gamma'].max())
    scatter = axes[1].scatter(df['frob_mean'], df['lambda_2'], 
                             c=df['gamma'], cmap='viridis', s=60, alpha=0.7, edgecolors='black')
    axes[1].set_xlabel('Mean Frobenius Norm')
    axes[1].set_ylabel('λ₂ (Algebraic Connectivity)')
    axes[1].set_title('Frobenius Norm vs Network Connectivity\\n(colored by γ)')
    axes[1].grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=axes[1], label='Gamma')
    
    plt.tight_layout()
    plt.show()
    
    # Print key correlations
    print("\\n=== Key Correlations ===")
    network_metrics = ['lambda_2', 'eigenratio', 'modularity', 'avg_path']
    decomp_metrics = ['frob_mean', 'cR_mean', 'cS_mean']
    
    for net_metric in network_metrics:
        print(f"\\n{net_metric}:")
        for decomp_metric in decomp_metrics:
            corr = correlation_matrix.loc[net_metric, decomp_metric]
            print(f"  vs {decomp_metric}: {corr:.3f}")
    
    return df

# Analyze relationships
analysis_df = analyze_decomposition_variable_relationships(connectivity_decomposition_data)

In [ ]:
def compare_decomposition_methods(connectivity_decomposition_data):
    """Compare different methods of creating coupling matrices from decomposition variables."""
    
    methods = ['correlation', 'distance', 'product', 'adjacency']
    
    # Take first model and layer 0 for comparison
    first_model = list(connectivity_decomposition_data.keys())[0]
    decomp = connectivity_decomposition_data[first_model]['decomposition'][0]
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    
    for i, method in enumerate(methods):
        # Create coupling matrix using this method
        coupling_matrix = create_coupling_matrix_from_decomposition(
            decomp['frob_norms'], decomp['c_R'], decomp['c_S'], method=method
        )
        
        # Visualize coupling matrix
        im1 = axes[0, i].imshow(coupling_matrix, cmap='RdBu_r', aspect='auto')
        axes[0, i].set_title(f'{method.capitalize()} Method')
        axes[0, i].set_xlabel('Node Index')
        axes[0, i].set_ylabel('Node Index')
        plt.colorbar(im1, ax=axes[0, i])
        
        # Compute and display network metrics
        try:
            metrics = compute_all_metrics(coupling_matrix, directed=False, threshold=1e-6)
            
            # Create a summary plot
            metric_values = [
                metrics['spectral']['lambda_2'],
                metrics['spectral']['eigenratio'],
                metrics['community']['modularity'],
                metrics['path']['avg_shortest_path']
            ]
            metric_names = ['λ₂', 'Eigenratio', 'Modularity', 'Avg Path']
            
            bars = axes[1, i].bar(metric_names, metric_values, alpha=0.7)
            axes[1, i].set_title(f'{method.capitalize()} - Network Metrics')
            axes[1, i].set_ylabel('Metric Value')
            
            # Add value labels on bars
            for bar, val in zip(bars, metric_values):
                height = bar.get_height()
                axes[1, i].text(bar.get_x() + bar.get_width()/2., height,
                               f'{val:.3f}', ha='center', va='bottom', fontsize=9)
                
        except Exception as e:
            axes[1, i].text(0.5, 0.5, f'Error: {str(e)[:30]}...', 
                           transform=axes[1, i].transAxes, ha='center', va='center')
    
    plt.suptitle(f'Comparison of Coupling Matrix Creation Methods\\nModel: {first_model}', \n                fontsize=16)
    plt.tight_layout()
    plt.show()

# Compare different methods
compare_decomposition_methods(connectivity_decomposition_data)

In [ ]:
def visualize_decomposition_networks(decomposition_networks):
    """Visualize network properties derived from connectivity decomposition."""
    
    # Collect data for plotting
    model_names = list(decomposition_networks.keys())
    gammas = [decomposition_networks[name]['gamma'] for name in model_names]
    Ts = [decomposition_networks[name]['T'] for name in model_names]
    
    # Extract metrics for each layer
    layers = [0, 1, 2]
    
    fig, axes = plt.subplots(3, 3, figsize=(18, 18))
    
    metrics_to_plot = ['lambda_2', 'eigenratio', 'modularity']
    metric_labels = ['λ₂ (Algebraic Connectivity)', 'Eigenratio (λ_N/λ₂)', 'Modularity Q']
    
    for layer_idx in layers:
        for metric_idx, (metric_key, label) in enumerate(zip(metrics_to_plot, metric_labels)):
            ax = axes[layer_idx, metric_idx]
            
            # Extract metric values for this layer
            values = []
            valid_gammas = []
            valid_Ts = []
            
            for name in model_names:
                networks = decomposition_networks[name]['networks']
                if layer_idx in networks and networks[layer_idx] is not None:
                    if metric_key == 'lambda_2':
                        val = networks[layer_idx]['metrics']['spectral']['lambda_2']
                    elif metric_key == 'eigenratio':
                        val = networks[layer_idx]['metrics']['spectral']['eigenratio']
                    elif metric_key == 'modularity':
                        val = networks[layer_idx]['metrics']['community']['modularity']
                    
                    values.append(val)
                    valid_gammas.append(decomposition_networks[name]['gamma'])
                    valid_Ts.append(decomposition_networks[name]['T'])
            
            if values:
                # Scatter plot colored by T values
                scatter = ax.scatter(valid_gammas, values, c=valid_Ts, cmap='viridis', 
                                   s=100, alpha=0.7, edgecolors='black', linewidth=0.5)
                
                ax.set_xlabel('Gamma', fontsize=12)
                ax.set_ylabel(label, fontsize=12)
                ax.set_title(f'Layer {layer_idx}: {label}', fontsize=14)
                ax.set_xscale('log')
                ax.grid(True, alpha=0.3)
                
                # Add colorbar for T values
                if metric_idx == 2:  # Only add colorbar to rightmost plots
                    cbar = plt.colorbar(scatter, ax=ax)
                    cbar.set_label('T Value', fontsize=12)
                
                # Annotate points with model names
                for i, name in enumerate(model_names):\n                    if i < len(values):\n                        ax.annotate(name.split()[0], \n                                   (valid_gammas[i], values[i]), \n                                   xytext=(5, 5), textcoords='offset points', \n                                   fontsize=8, alpha=0.8)
            else:
                ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, \n                       ha='center', va='center', fontsize=14)
    
    plt.suptitle('Network Metrics from Connectivity Decomposition\\n(frob_norms, c_R, c_S)', \n                fontsize=16, y=0.98)
    plt.tight_layout()
    plt.show()

# Visualize the results
visualize_decomposition_networks(decomposition_networks)

In [ ]:
def create_coupling_matrix_from_decomposition(frob_norms, c_R, c_S, method='correlation'):
    """
    Create coupling matrices from decomposition variables for network analysis.
    
    Different methods to convert decomposition variables to coupling matrices:
    1. 'correlation': Use correlation matrix between variables
    2. 'distance': Use distance-based coupling 
    3. 'product': Use outer product of variables
    4. 'adjacency': Create adjacency based on thresholds
    """
    
    n = len(frob_norms)
    
    if method == 'correlation':
        # Stack variables and compute correlation matrix
        variables = np.stack([frob_norms, c_R, c_S], axis=0)
        # Add small noise to avoid perfect correlations
        variables += np.random.normal(0, 1e-6, variables.shape)
        coupling_matrix = np.corrcoef(variables.T)
        
    elif method == 'distance':
        # Create distance-based coupling matrix
        points = np.stack([frob_norms, c_R, c_S], axis=1)
        # Compute pairwise distances
        dist_matrix = np.linalg.norm(points[:, None] - points[None, :], axis=2)
        # Convert to coupling (higher for closer points)
        coupling_matrix = 1.0 / (1.0 + dist_matrix)
        np.fill_diagonal(coupling_matrix, 0)  # Remove self-coupling
        
    elif method == 'product':
        # Use outer product weighted by different components
        frob_outer = np.outer(frob_norms, frob_norms)
        cR_outer = np.outer(c_R, c_R) 
        cS_outer = np.outer(c_S, c_S)
        coupling_matrix = 0.4 * frob_outer + 0.3 * cR_outer + 0.3 * cS_outer
        np.fill_diagonal(coupling_matrix, 0)
        
    elif method == 'adjacency':
        # Create binary adjacency based on similarity thresholds
        coupling_matrix = np.zeros((n, n))
        
        # Define similarity based on relative differences
        frob_thresh = np.std(frob_norms) * 0.5
        cR_thresh = np.std(c_R) * 0.5
        cS_thresh = np.std(c_S) * 0.5
        
        for i in range(n):
            for j in range(i+1, n):
                frob_sim = abs(frob_norms[i] - frob_norms[j]) < frob_thresh
                cR_sim = abs(c_R[i] - c_R[j]) < cR_thresh
                cS_sim = abs(c_S[i] - c_S[j]) < cS_thresh
                
                # Connect if similar in at least 2 out of 3 metrics
                if sum([frob_sim, cR_sim, cS_sim]) >= 2:
                    coupling_matrix[i, j] = 1.0
                    coupling_matrix[j, i] = 1.0
    
    return coupling_matrix

def analyze_decomposition_networks(connectivity_decomposition_data, method='correlation'):
    """Perform network analysis on connectivity decomposition variables."""
    
    network_results = {}
    
    for name, data in connectivity_decomposition_data.items():
        print(f"\\n=== Network Analysis for {name} ===")
        gamma = data['gamma']
        T = data['T']
        
        model_networks = {}
        
        for layer_idx, decomp in data['decomposition'].items():
            print(f"\\nLayer {layer_idx}:")
            
            # Extract variables
            frob_norms = decomp['frob_norms']
            c_R = decomp['c_R']
            c_S = decomp['c_S']
            
            # Create coupling matrix
            coupling_matrix = create_coupling_matrix_from_decomposition(
                frob_norms, c_R, c_S, method=method
            )
            
            print(f"  Created {coupling_matrix.shape[0]}×{coupling_matrix.shape[1]} coupling matrix")
            print(f"  Coupling range: [{coupling_matrix.min():.4f}, {coupling_matrix.max():.4f}]")
            
            # Compute network metrics
            try:
                metrics = compute_all_metrics(coupling_matrix, directed=False, threshold=1e-6)
                
                # Print key metrics
                print(f"  Spectral: λ₂={metrics['spectral']['lambda_2']:.6f}, ratio={metrics['spectral']['eigenratio']:.2f}")
                print(f"  Modularity: Q={metrics['community']['modularity']:.4f}")
                print(f"  Path length: {metrics['path']['avg_shortest_path']:.4f}")
                
                model_networks[layer_idx] = {
                    'coupling_matrix': coupling_matrix,
                    'metrics': metrics,
                    'decomposition': decomp
                }
                
            except Exception as e:
                print(f"  Error computing metrics: {e}")
                model_networks[layer_idx] = None
        
        network_results[name] = {
            'networks': model_networks,
            'gamma': gamma,
            'T': T
        }
    
    return network_results

# Analyze networks using different methods
print("Analyzing decomposition networks using correlation method...")
decomposition_networks = analyze_decomposition_networks(connectivity_decomposition_data, method='correlation')

In [ ]:
def extract_connectivity_decomposition(model, layer_idx=0):
    """Extract connectivity decomposition results for network analysis."""
    
    # Create static analyzer for the specified layer
    static_analyzer = AKOrNStaticAnalyzer(model, layer_idx, device=device)
    
    # Extract connectivity blocks
    connectivity_blocks = static_analyzer.extract_connectivity_blocks()
    if connectivity_blocks is None:
        print(f"Could not extract connectivity blocks for layer {layer_idx}")
        return None
    
    print(f"Layer {layer_idx}: Extracted {len(connectivity_blocks)} connectivity blocks")
    print(f"  Block shape: {connectivity_blocks[0].shape}")
    
    # Compute decomposition metrics
    results = {}
    
    # 1. Frobenius norms (line 392 in analysis_utils.py)
    frob_norms = np.linalg.norm(connectivity_blocks, axis=(1, 2))
    results['frob_norms'] = frob_norms
    
    # 2. Rotation/Symmetric decomposition (c_R, c_S)
    c_R, c_S, alpha, beta = static_analyzer.decompose_rotation_symmetric(connectivity_blocks)
    results['c_R'] = c_R
    results['c_S'] = c_S
    results['alpha'] = alpha
    results['beta'] = beta
    
    # 3. Symmetric/Skew-symmetric decomposition  
    p1, p2, p3, q, sym_frob, skew_frob = static_analyzer.decompose_symmetric_skew(connectivity_blocks)
    results['sym_frob'] = sym_frob
    results['skew_frob'] = skew_frob
    
    print(f"  Decomposition results:")
    print(f"    Frob norms: mean={frob_norms.mean():.4f}, std={frob_norms.std():.4f}")
    print(f"    c_R: mean={c_R.mean():.4f}, std={c_R.std():.4f}")
    print(f"    c_S: mean={c_S.mean():.4f}, std={c_S.std():.4f}")
    
    return results

# Extract connectivity decomposition for all loaded models
connectivity_decomposition_data = {}

for name, model_data in loaded_models.items():
    print(f"\\nExtracting connectivity decomposition for {name}...")
    model = model_data["model"]
    
    layer_results = {}
    for layer_idx in range(3):  # Analyze all 3 layers
        decomp_results = extract_connectivity_decomposition(model, layer_idx)
        if decomp_results is not None:
            layer_results[layer_idx] = decomp_results
    
    connectivity_decomposition_data[name] = {
        'decomposition': layer_results,
        'gamma': model_data['gamma'],
        'T': model_data['T']
    }

print(f"\\nExtracted connectivity decomposition for {len(connectivity_decomposition_data)} models")

In [ ]:
# Import network analysis utilities
from source.kuramoto_network_metrics import (
    compute_all_metrics, 
    spectral_metrics, 
    strength_metrics,
    community_metrics,
    path_metrics,
    graph_from_K
)
from source.models.classification.analysis_utils import AKOrNStaticAnalyzer
import networkx as nx
import pandas as pd

print("Network analysis utilities imported successfully!")

## Network Analysis of Connectivity Decomposition

Now let's analyze the connectivity decomposition results (frob_norms, c_R, c_S) from a network perspective using the Kuramoto network metrics.